# k-Nearest Neighbors Taxi Destination Notebook

Cached-data Kaggle workflow for Porto taxi destination prediction: load prebuilt preprocessing artifacts generated by `code/build_preprocessed_datasets.py`, train a k-Nearest Neighbors model, validate with Haversine distance, and export a submission.


## Cached Data Paths

These model notebooks do not rebuild raw preprocessing. They expect the cached preprocessed Kaggle Dataset generated by `code/build_preprocessed_datasets.py`, including `metadata.json`, `validation/`, and `final/` matrices. Attach that dataset on Kaggle or set `PREPROCESSED_DATA_DIR`; model artifacts and submissions are written under `/kaggle/working`.


In [ ]:
import os
from pathlib import Path

PREPROCESSED_DATA_DIR = Path(
    os.environ.get(
        "PREPROCESSED_DATA_DIR",
        "/kaggle/input/datasets/louishogge/preprocessed-taxi-dataset/data",
    )
)
OUTPUT_DIR = Path("/kaggle/working")
SUBMISSION_DIR = OUTPUT_DIR / "submission"
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Preprocessed data directory:", PREPROCESSED_DATA_DIR)
print("Output directory:", OUTPUT_DIR.resolve())
print("Submission directory:", SUBMISSION_DIR.resolve())
print("Artifact directory:", ARTIFACT_DIR.resolve())


## Imports

Load dependencies used by cached data loading, modeling, tuning, and plots.


In [ ]:
import gc
import json
import os
import random
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterGrid

warnings.filterwarnings("ignore")


## Hyperparameters

Set the reproducible run configuration, accepted model parameters, and reviewed run provenance.


In [ ]:
SEED = 42
MODEL_NAME = "k-Nearest Neighbors"
MODEL_SLUG = "knn"

RUN_HYPERPARAMETER_TUNING = False
# Keep tuning below Kaggle's 12-hour consecutive run limit.
TUNING_MAX_TRIALS = None
TUNING_MAX_SECONDS = 11 * 60 * 60
TUNING_DEADLINE_SAFETY_SECONDS = 30 * 60

TUNING_TRAIN_SAMPLE_SIZE = 100_000
TUNING_VAL_SAMPLE_SIZE = 10_000
PROMOTION_TOP_N = 5
PROMOTION_TRAIN_SAMPLE_SIZE = 200_000
PROMOTION_VAL_SAMPLE_SIZE = 20_000
PREDICT_BATCH_SIZE = 256
FINAL_TRAIN_SAMPLE_SIZE = None
SERIALIZE_KNN_MODEL = False

WANDB_PROJECT = "machine-learning"
WANDB_ENTITY = "ml-l"
WANDB_SECRET_NAME = "wandb-api-key"

SUBMISSION_FILE = f"submission_{MODEL_SLUG}.csv"
BEST_PARAMS_PATH = ARTIFACT_DIR / f"best_params_{MODEL_SLUG}.json"
MODEL_BUNDLE_PATH = ARTIFACT_DIR / f"model_bundle_{MODEL_SLUG}.joblib"

BEST_MODEL_PARAMS = {
    "n_neighbors": 256,
    "weights": "distance",
    "metric": "manhattan",
    "algorithm": "brute",
    "n_jobs": -1,
}

BEST_MODEL_RUN_SUMMARY = {
    "reviewed_on": "2026-07-09",
    "source": "W&B finite-search trial 24 and promotion trial 24 plus local submission scoring",
    "trial_index": 24,
    "equivalent_best_trial_indices": [24],
    "tuner_type": "finite_search_promotion",
    "tuning_stage": "promotion",
    "historical_unique_candidate_count": 42,
    "historical_tuning_train_sample_size": 20_000,
    "historical_tuning_val_sample_size": 3_000,
    "historical_promotion_train_sample_size": 50_000,
    "historical_promotion_val_sample_size": 5_000,
    "search_sample_haversine_mean_km": 3.5033793566382156,
    "validation_haversine_mean_km": 3.5655072471257094,
    "validation_haversine_median_km": 2.537559503534283,
    "validation_haversine_std_km": 4.379586975227966,
    "validation_haversine_p90_km": 6.9971918746731125,
    "fit_seconds": 0.6060214042663574,
    "predict_seconds": 102.23210167884827,
    "tuning_elapsed_seconds": 3621.018406867981,
    "local_submission_all_mean_km": 3.4597120791174008,
    "local_submission_all_median_km": 2.4511267444836395,
    "local_submission_all_std_km": 3.932310305576266,
    "local_submission_all_p90_km": 6.88156324633888,
    "local_submission_public_mean_km": 3.351110061100981,
    "local_submission_private_mean_km": 3.5683140971338205,
    "local_submission_n_trips": 320,
}

TUNING_SEARCH_SPACE = {
    "n_neighbors": [192, 224, 256, 288, 320],
    "weights": ["uniform", "distance"],
    "metric": ["manhattan"],
    "algorithm": ["brute"],
    "n_jobs": [-1],
}

MODEL_PARAMS = BEST_MODEL_PARAMS.copy()
TUNING_RUNTIME_STATE = {
    "unique_candidate_count": None,
    "tuning_elapsed_seconds": None,
    "selected_metric_source": None,
}


## Reproducibility

Seed random generators used by model training and tuning.


In [ ]:
def seed_everything(seed=SEED):
    """Seed Python and NumPy random generators."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)
print("Seed:", SEED)


## Utility Constants

Define the cached target-column order used by the preprocessing helpers.

In [ ]:
TARGET_COLUMNS = ["END_LONG", "END_LAT"]

## Cached Preprocessed Dataset Helpers

Load and validate the shared preprocessed validation and final datasets.


In [ ]:
PREPROCESSED_SCHEMA_VERSION = 1
EXPECTED_CATEGORICAL_ENCODING_POLICY = {
    "CALL_TYPE": "one_hot",
    "ORIGIN_CALL": "missing_indicator_plus_log_count",
    "ORIGIN_STAND": "one_hot_missing_as_0",
    "TAXI_ID": "one_hot",
}


def require_existing_path(path, description):
    """Return an existing path or fail with an actionable Kaggle Dataset message."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}. "
            "Attach the preprocessed-taxi-dataset Kaggle Dataset or set PREPROCESSED_DATA_DIR."
        )
    return path


def load_preprocessed_metadata(base_dir=PREPROCESSED_DATA_DIR):
    """Load and validate cached preprocessing metadata."""
    metadata_path = require_existing_path(base_dir / "metadata.json", "preprocessed metadata")
    with open(metadata_path, "r", encoding="utf-8") as file:
        metadata = json.load(file)

    schema_version = metadata.get("schema_version")
    if schema_version != PREPROCESSED_SCHEMA_VERSION:
        raise ValueError(
            f"Unsupported preprocessed dataset schema_version={schema_version!r}; "
            f"expected {PREPROCESSED_SCHEMA_VERSION}."
        )

    categorical_encoding_policy = metadata.get("constants", {}).get("categorical_encoding_policy")
    if categorical_encoding_policy != EXPECTED_CATEGORICAL_ENCODING_POLICY:
        raise ValueError(
            "Attached preprocessed dataset was built with an unexpected categorical encoding policy. "
            f"Expected {EXPECTED_CATEGORICAL_ENCODING_POLICY!r}, got {categorical_encoding_policy!r}. "
            "Regenerate the cached dataset with the current build_preprocessed_datasets.py script."
        )
    return metadata


def load_cached_frame(relative_path, base_dir=PREPROCESSED_DATA_DIR):
    """Load one cached Parquet frame from the preprocessed dataset."""
    frame_path = require_existing_path(base_dir / relative_path, f"cached frame {relative_path}")
    return pd.read_parquet(frame_path)


def assert_matching_features(left, right, left_name, right_name):
    """Validate that two feature matrices use the same columns in the same order."""
    if not left.columns.equals(right.columns):
        missing_from_right = list(left.columns.difference(right.columns))
        missing_from_left = list(right.columns.difference(left.columns))
        raise ValueError(
            f"Feature columns differ between {left_name} and {right_name}. "
            f"Missing from {right_name}: {missing_from_right}; missing from {left_name}: {missing_from_left}."
        )


def assert_index_aligned(features, targets, split_name):
    """Validate feature/target row alignment."""
    if not features.index.equals(targets.index):
        raise ValueError(f"Cached {split_name} features and targets have different indexes.")


def assert_target_columns(targets, split_name):
    """Validate target columns and order."""
    if list(targets.columns) != TARGET_COLUMNS:
        raise ValueError(
            f"Cached {split_name} target columns are {list(targets.columns)!r}; "
            f"expected {TARGET_COLUMNS!r}."
        )


def load_cached_validation_data():
    """Load cached train/validation matrices for fair model comparison."""
    metadata = load_preprocessed_metadata()
    validation_dir = require_existing_path(PREPROCESSED_DATA_DIR / "validation", "validation data directory")

    X_train = load_cached_frame("validation/X_train.parquet")
    y_train = load_cached_frame("validation/y_train.parquet")
    X_val = load_cached_frame("validation/X_val.parquet")
    y_val = load_cached_frame("validation/y_val.parquet")

    assert_index_aligned(X_train, y_train, "validation train")
    assert_index_aligned(X_val, y_val, "validation holdout")
    assert_matching_features(X_train, X_val, "X_train", "X_val")
    assert_target_columns(y_train, "validation train")
    assert_target_columns(y_val, "validation holdout")

    print("Loaded cached validation data from:", validation_dir)
    print("X_train shape:", X_train.shape)
    print("y_train shape:", y_train.shape)
    print("X_val shape:", X_val.shape)
    print("y_val shape:", y_val.shape)
    print("Feature column count:", X_train.shape[1])
    return metadata, X_train, y_train, X_val, y_val


def load_cached_final_data():
    """Load cached full-train and competition-test matrices for submission."""
    final_dir = require_existing_path(PREPROCESSED_DATA_DIR / "final", "final data directory")
    scaler_path = require_existing_path(final_dir / "gps_scaler.joblib", "final GPS scaler")
    categorical_encoder_path = require_existing_path(
        final_dir / "categorical_encoder.joblib",
        "final categorical encoder",
    )

    X_final_train = load_cached_frame("final/X_train.parquet")
    y_final_train = load_cached_frame("final/y_train.parquet")
    X_final_test = load_cached_frame("final/X_test.parquet")
    final_gps_scaler = joblib.load(scaler_path)
    final_categorical_encoder = joblib.load(categorical_encoder_path)

    assert_index_aligned(X_final_train, y_final_train, "final train")
    assert_matching_features(X_final_train, X_final_test, "X_final_train", "X_final_test")
    assert_target_columns(y_final_train, "final train")

    print("Loaded cached final data from:", final_dir)
    print("X_final_train shape:", X_final_train.shape)
    print("y_final_train shape:", y_final_train.shape)
    print("X_final_test shape:", X_final_test.shape)
    return X_final_train, y_final_train, X_final_test, final_gps_scaler, final_categorical_encoder


## Evaluation And Submission Helpers

Evaluate validation predictions and write the final Kaggle submission file.


In [ ]:
def haversine(pred, gt):
    """Return row-wise Haversine distances in kilometers for [LAT, LONG] arrays."""
    pred_lat = np.radians(pred[:, 0])
    pred_long = np.radians(pred[:, 1])
    gt_lat = np.radians(gt[:, 0])
    gt_long = np.radians(gt[:, 1])

    dlat = gt_lat - pred_lat
    dlon = gt_long - pred_long

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(pred_lat) * np.cos(gt_lat) * np.sin(dlon / 2) ** 2
    )

    d = 2 * 6371 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    return d

def long_lat_to_lat_long(values):
    """Convert [LONG, LAT] arrays to [LAT, LONG] arrays."""
    values = np.asarray(values, dtype=float)
    return np.column_stack([values[:, 1], values[:, 0]])

def evaluate_long_lat_predictions(y_true_long_lat, y_pred_long_lat, prefix="val"):
    """Evaluate [LONG, LAT] predictions with Haversine distances."""
    distances = haversine(
        pred=long_lat_to_lat_long(y_pred_long_lat),
        gt=long_lat_to_lat_long(y_true_long_lat),
    )
    return {
        f"{prefix}/haversine_mean_km": float(distances.mean()),
        f"{prefix}/haversine_median_km": float(np.median(distances)),
        f"{prefix}/haversine_std_km": float(distances.std()),
        f"{prefix}/haversine_p90_km": float(np.percentile(distances, 90)),
    }, distances

def write_submission(trip_ids, destinations, output_path):
    """Write a Kaggle submission CSV from [LAT, LONG] destinations."""
    n_samples = len(trip_ids)
    assert destinations.shape == (n_samples, 2)

    submission = pd.DataFrame(
        data={
            'LATITUDE': destinations[:, 0],
            'LONGITUDE': destinations[:, 1],
        },
        columns=["LATITUDE", "LONGITUDE"],
        index=trip_ids,
    )

    submission.to_csv(output_path, index_label="TRIP_ID")


def make_json_safe(value):
    """Convert common notebook values to JSON-safe objects."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, (float, np.floating)):
        value = float(value)
        return value if np.isfinite(value) else None
    if isinstance(value, np.integer):
        return value.item()
    if isinstance(value, np.ndarray):
        return make_json_safe(value.tolist())
    if isinstance(value, pd.Series):
        return make_json_safe(value.to_dict())
    if isinstance(value, pd.DataFrame):
        return make_json_safe(value.to_dict(orient="records"))
    if isinstance(value, dict):
        return {str(key): make_json_safe(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]
    return value

def save_json_artifact(payload, output_path):
    """Write an indented JSON artifact to Kaggle output storage."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(make_json_safe(payload), file, indent=2, sort_keys=True, allow_nan=False)

def get_runtime_profile():
    """Return the KNN runtime profile used to keep Kaggle runs bounded."""
    return {
        "tuning_strategy": "finite_search",
        "tuning_train_sample_size": TUNING_TRAIN_SAMPLE_SIZE,
        "tuning_val_sample_size": TUNING_VAL_SAMPLE_SIZE,
        "promotion_top_n": PROMOTION_TOP_N,
        "promotion_train_sample_size": PROMOTION_TRAIN_SAMPLE_SIZE,
        "promotion_val_sample_size": PROMOTION_VAL_SAMPLE_SIZE,
        "predict_batch_size": PREDICT_BATCH_SIZE,
        "final_train_sample_size": FINAL_TRAIN_SAMPLE_SIZE,
        "serialize_knn_model": SERIALIZE_KNN_MODEL,
        "tuning_deadline_safety_seconds": TUNING_DEADLINE_SAFETY_SECONDS,
        "unique_candidate_count": TUNING_RUNTIME_STATE["unique_candidate_count"],
        "tuning_elapsed_seconds": TUNING_RUNTIME_STATE["tuning_elapsed_seconds"],
        "selected_metric_source": TUNING_RUNTIME_STATE["selected_metric_source"],
    }

def save_best_params_artifact(tuning_results, validation_metrics):
    """Persist active parameters and their sampled validation context."""
    payload = {
        "model_name": MODEL_NAME,
        "model_slug": MODEL_SLUG,
        "params_source": "tuning_results" if not tuning_results.empty else "BEST_MODEL_PARAMS",
        "saved_best_model_params": BEST_MODEL_PARAMS,
        "best_model_run_summary": BEST_MODEL_RUN_SUMMARY,
        "active_model_params": MODEL_PARAMS,
        "validation_metrics": validation_metrics,
        "runtime_profile": get_runtime_profile(),
        "preprocessed_dataset": {
            "path": str(PREPROCESSED_DATA_DIR),
            "schema_version": PREPROCESSED_METADATA.get("schema_version"),
            "config": PREPROCESSED_METADATA.get("config", {}),
            "validation": PREPROCESSED_METADATA.get("validation", {}),
            "final": PREPROCESSED_METADATA.get("final", {}),
        },
        "tuning": {
            "run_hyperparameter_tuning": RUN_HYPERPARAMETER_TUNING,
            "strategy": "finite_search",
            "max_trials": TUNING_MAX_TRIALS,
            "max_seconds": TUNING_MAX_SECONDS,
            "search_space": TUNING_SEARCH_SPACE,
        },
    }
    if not tuning_results.empty:
        sort_key = "selection_metric_value" if "selection_metric_value" in tuning_results.columns else "val_sample/haversine_mean_km"
        if "tuning_stage" in tuning_results.columns and (tuning_results["tuning_stage"] == "promotion").any():
            best_trial_pool = tuning_results[tuning_results["tuning_stage"] == "promotion"]
        else:
            best_trial_pool = tuning_results
        best_trial = best_trial_pool.sort_values(sort_key, ascending=True).iloc[0]
        payload["best_trial"] = best_trial.to_dict()

    save_json_artifact(payload, BEST_PARAMS_PATH)
    return payload


## Cached Validation Dataset Loading

Load shared train/validation matrices for fair model comparison.


In [ ]:
PREPROCESSED_METADATA, X_train, y_train, X_val, y_val = load_cached_validation_data()

validation_metadata = PREPROCESSED_METADATA.get("validation", {})
prefix_summary = pd.DataFrame(validation_metadata.get("prefix_summary", {}))
if not prefix_summary.empty:
    display(prefix_summary)


## Define k-Nearest Neighbors Model

Build the estimator used by validation, tuning, and final submission.


In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def sample_aligned_xy(X, y, n, label):
    """Sample aligned feature/target rows, stratifying by PREFIX_LENGTH when possible."""
    if n is None or n >= len(X):
        print(f"{label}: using all {len(X)} rows.")
        return X, y

    if n <= 0:
        raise ValueError(f"{label}: sample size must be positive or None, got {n!r}.")

    if not X.index.equals(y.index):
        raise ValueError(f"{label}: feature and target indexes are not aligned.")

    if "PREFIX_LENGTH" not in X.columns:
        sampled_index = X.sample(n=n, random_state=SEED).index
    else:
        groups = X.groupby("PREFIX_LENGTH", sort=True).indices
        quotas = []
        for _, positions in groups.items():
            group_size = len(positions)
            exact = n * group_size / len(X)
            base = min(group_size, int(np.floor(exact)))
            quotas.append({"positions": np.asarray(positions), "base": base, "fraction": exact - base})

        remaining = n - sum(item["base"] for item in quotas)
        for item in sorted(quotas, key=lambda row: row["fraction"], reverse=True):
            if remaining <= 0:
                break
            if item["base"] < len(item["positions"]):
                item["base"] += 1
                remaining -= 1

        sampled_parts = []
        rng = np.random.default_rng(SEED)
        for item in quotas:
            take = item["base"]
            if take <= 0:
                continue
            chosen_positions = rng.choice(item["positions"], size=take, replace=False)
            sampled_parts.append(X.index[chosen_positions])

        sampled_index = pd.Index(np.concatenate([part.to_numpy() for part in sampled_parts]))
        if len(sampled_index) > n:
            sampled_index = sampled_index.to_series().sample(n=n, random_state=SEED).index
        elif len(sampled_index) < n:
            missing = X.index.difference(sampled_index)
            fill = X.loc[missing].sample(n=n - len(sampled_index), random_state=SEED).index
            sampled_index = sampled_index.append(fill)

    sampled_index = pd.Index(sampled_index).sort_values()
    print(f"{label}: sampled {len(sampled_index)} of {len(X)} rows.")
    return X.loc[sampled_index].copy(), y.loc[sampled_index].copy()


def predict_in_batches(model, X, batch_size=PREDICT_BATCH_SIZE, label="prediction"):
    """Predict in bounded batches and log progress for long KNN distance scans."""
    if batch_size is None or batch_size <= 0:
        batch_size = len(X)

    start_time = time.time()
    predictions = []
    n_rows = len(X)
    n_batches = int(np.ceil(n_rows / batch_size)) if n_rows else 0
    print(f"{label}: predicting {n_rows} rows in {n_batches} batch(es).")

    for batch_number, start in enumerate(range(0, n_rows, batch_size), start=1):
        stop = min(start + batch_size, n_rows)
        predictions.append(model.predict(X.iloc[start:stop]))
        if batch_number == 1 or batch_number == n_batches or batch_number % 10 == 0:
            elapsed = time.time() - start_time
            print(f"{label}: batch {batch_number}/{n_batches} rows {start}:{stop} elapsed={elapsed:.1f}s")

    if len(predictions) == 0:
        return np.empty((0, len(TARGET_COLUMNS)))
    return np.vstack(predictions)


def build_model(params=None, n_training_samples=None):
    """Build the k-Nearest Neighbors estimator with active parameters."""
    model_params = dict(params or MODEL_PARAMS)
    model_params["n_neighbors"] = int(model_params["n_neighbors"])
    if n_training_samples is not None and model_params["n_neighbors"] > n_training_samples:
        model_params["n_neighbors"] = n_training_samples
        print(f"Capped n_neighbors to {n_training_samples} for this sample-sized run.")

    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(**model_params)),
    ])


## Hyperparameter Tuning

Optional tuning is disabled by default. When enabled, a staged finite search scores candidates on deterministic samples so the search can complete on Kaggle.


### Tuning Logging Helpers

Validate W&B once before optional tuning. Missing or invalid credentials disable logging, and per-trial SDK failures never interrupt model evaluation.

In [ ]:
WANDB_ENABLED = False
_WANDB_MODULE = None
_WANDB_MODE = "disabled"

def get_wandb_key():
    """Return the W&B API key from Kaggle secrets or the environment."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(WANDB_SECRET_NAME)
    except Exception:
        return os.environ.get("WANDB_API_KEY")

def _wandb_warning(stage, exception):
    """Report a sanitized W&B failure without exposing credentials."""
    print(
        f"W&B {stage} failed ({type(exception).__name__}). "
        "Model tuning will continue."
    )

def initialize_wandb():
    """Validate W&B credentials once before optional tuning starts."""
    wandb_key = get_wandb_key()
    if not wandb_key:
        print("No W&B key found. Tuning will run without W&B logging.")
        return None, "disabled", False

    try:
        import wandb
        login_succeeded = wandb.login(key=wandb_key)
        if login_succeeded is False:
            raise RuntimeError("W&B login was rejected.")
    except Exception as exception:
        _wandb_warning("setup", exception)
        return None, "disabled", False

    return wandb, "online", True

def sanitize_for_wandb(value):
    """Convert values to W&B-safe scalar or container types."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, dict):
        return {key: sanitize_for_wandb(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [sanitize_for_wandb(item) for item in value]
    return value

def log_tuning_trial_to_wandb(result):
    """Log one tuning trial without allowing W&B to interrupt tuning."""
    if not WANDB_ENABLED:
        return

    run = None
    try:
        trial_index = int(result["trial_index"])
        params = sanitize_for_wandb(result["params"])
        tuner_type = result["tuner_type"]
        preprocessed_config = PREPROCESSED_METADATA.get("config", {})
        config = {
            "model_slug": MODEL_SLUG,
            "trial_index": trial_index,
            "trial_params": params,
            "seed": SEED,
            "tuner_type": tuner_type,
            "preprocessed_schema_version": PREPROCESSED_METADATA.get("schema_version"),
            "preprocessed_n_rows": preprocessed_config.get("n_rows"),
            "preprocessed_validation_size": preprocessed_config.get("validation_size"),
            "preprocessed_prefix_sample_factor": preprocessed_config.get("prefix_sample_factor"),
            "tuning_strategy": "finite_search",
            "runtime_profile": sanitize_for_wandb(get_runtime_profile()),
        }

        metrics = {
            key: sanitize_for_wandb(value)
            for key, value in result.items()
            if key not in {"params", "params_repr", "tuner_type", "selection_metric_name", "tuning_stage"}
        }

        run = _WANDB_MODULE.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            name=f"{MODEL_SLUG}-{tuner_type}-trial-{trial_index:03d}",
            mode=_WANDB_MODE,
            config=config,
            reinit=True,
        )
        if run is None:
            raise RuntimeError("wandb.init returned no run.")
        _WANDB_MODULE.log(metrics)
    except Exception as exception:
        _wandb_warning("trial logging", exception)
    finally:
        if run is not None:
            try:
                run.finish()
            except Exception as exception:
                _wandb_warning("run finalization", exception)

if RUN_HYPERPARAMETER_TUNING:
    _WANDB_MODULE, _WANDB_MODE, WANDB_ENABLED = initialize_wandb()

### Tuning Trial Evaluation

Fit candidate configurations and rank them by validation Haversine distance.


In [ ]:
def evaluate_tuning_trial(
    trial_params,
    trial_index,
    tuner_type,
    X_fit,
    y_fit,
    X_eval,
    y_eval,
    eval_prefix,
    tuning_stage,
):
    """Fit one sampled KNN trial and return sampled validation metrics."""
    trial_params = dict(trial_params)
    trial_model = build_model(params=trial_params, n_training_samples=len(X_fit))

    start_time = time.time()
    trial_model.fit(X_fit, y_fit)
    fit_seconds = time.time() - start_time

    pred_start_time = time.time()
    eval_predictions = predict_in_batches(
        trial_model,
        X_eval,
        batch_size=PREDICT_BATCH_SIZE,
        label=f"{tuner_type} trial {trial_index} {eval_prefix}",
    )
    predict_seconds = time.time() - pred_start_time

    eval_metrics, _ = evaluate_long_lat_predictions(y_eval.values, eval_predictions, prefix=eval_prefix)
    selection_metric_name = f"{eval_prefix}/haversine_mean_km"

    return {
        "trial_index": trial_index,
        "params": trial_params.copy(),
        "params_repr": repr(trial_params),
        "tuner_type": tuner_type,
        "tuning_stage": tuning_stage,
        "fit_rows": int(len(X_fit)),
        "eval_rows": int(len(X_eval)),
        **eval_metrics,
        "selection_metric_name": selection_metric_name,
        "selection_metric_value": float(eval_metrics[selection_metric_name]),
        "fit_seconds": float(fit_seconds),
        "predict_seconds": float(predict_seconds),
    }

def register_tuning_result(results, result):
    """Store, log, and print one sampled tuning result."""
    previous_best = min(
        [row["selection_metric_value"] for row in results if row["tuning_stage"] == result["tuning_stage"]],
        default=float("inf"),
    )
    result["trial/is_best_so_far"] = bool(result["selection_metric_value"] <= previous_best)
    results.append(result)
    log_tuning_trial_to_wandb(result)
    print(
        f"  stage={result['tuning_stage']}, "
        f"{result['selection_metric_name']}={result['selection_metric_value']:.4f} km, "
        f"fit_rows={result['fit_rows']}, eval_rows={result['eval_rows']}, "
        f"fit={result['fit_seconds']:.1f}s, predict={result['predict_seconds']:.1f}s"
    )

def finalize_tuning_results(results):
    """Select the best sampled tuning result and update MODEL_PARAMS."""
    global MODEL_PARAMS

    if len(results) == 0:
        raise RuntimeError("No hyperparameter trials were evaluated.")

    promotion_results = [row for row in results if row.get("tuning_stage") == "promotion"]
    selection_pool = promotion_results if promotion_results else results
    best_result = min(selection_pool, key=lambda row: row["selection_metric_value"])
    MODEL_PARAMS = best_result["params"].copy()
    TUNING_RUNTIME_STATE["selected_metric_source"] = best_result["selection_metric_name"]

    tuning_results = pd.DataFrame([
        {key: value for key, value in result.items() if key != "params"}
        for result in results
    ]).sort_values(["tuning_stage", "selection_metric_value"], ascending=[True, True])

    print(f"Best sampled validation Haversine: {best_result['selection_metric_value']:.6f} km")
    print(f"Selected from: {best_result['selection_metric_name']} ({best_result['tuning_stage']})")

    import pprint
    print("Paste-ready best config:")
    print("BEST_MODEL_PARAMS = " + pprint.pformat(MODEL_PARAMS, sort_dicts=False))
    display(tuning_results)
    return tuning_results


### Staged Finite Search

Evaluate the bounded candidate grid, then promote the strongest configurations to larger deterministic samples.


In [ ]:
def get_finite_candidate_configs():
    """Return the configured finite-grid candidate configurations."""
    candidates = list(ParameterGrid(TUNING_SEARCH_SPACE))
    if TUNING_MAX_TRIALS is not None:
        candidates = candidates[:TUNING_MAX_TRIALS]
    return candidates

def tuning_time_remaining(tuning_start_time):
    """Return remaining seconds before the protected tuning deadline."""
    if TUNING_MAX_SECONDS is None:
        return float("inf")
    protected_budget = max(0, TUNING_MAX_SECONDS - TUNING_DEADLINE_SAFETY_SECONDS)
    return protected_budget - (time.time() - tuning_start_time)

def should_stop_tuning(tuning_start_time):
    """Avoid starting expensive KNN work after the protected deadline."""
    remaining = tuning_time_remaining(tuning_start_time)
    if remaining <= 0:
        print(
            "Stopping tuning before the next KNN fit/predict because "
            f"the protected deadline was reached; safety buffer={TUNING_DEADLINE_SAFETY_SECONDS}s."
        )
        return True
    return False

def run_finite_search_tuning():
    """Run staged sampled finite search for KNN under Kaggle time limits."""
    candidates = get_finite_candidate_configs()
    TUNING_RUNTIME_STATE["unique_candidate_count"] = len(candidates)
    if len(candidates) == 0:
        raise ValueError("TUNING_SEARCH_SPACE produced no trials.")

    X_tuning_train, y_tuning_train = sample_aligned_xy(
        X_train,
        y_train,
        TUNING_TRAIN_SAMPLE_SIZE,
        "tuning train sample",
    )
    X_tuning_val, y_tuning_val = sample_aligned_xy(
        X_val,
        y_val,
        TUNING_VAL_SAMPLE_SIZE,
        "tuning validation sample",
    )

    print(f"Starting finite_search tuning for {MODEL_NAME} with {len(candidates)} candidate(s).")
    tuning_start_time = time.time()
    results = []

    for trial_index, trial_params in enumerate(candidates, start=1):
        if should_stop_tuning(tuning_start_time):
            break

        print(f"Trial {trial_index}/{len(candidates)}: {trial_params}")
        result = evaluate_tuning_trial(
            trial_params=trial_params,
            trial_index=trial_index,
            tuner_type="finite_search",
            X_fit=X_tuning_train,
            y_fit=y_tuning_train,
            X_eval=X_tuning_val,
            y_eval=y_tuning_val,
            eval_prefix="val_sample",
            tuning_stage="search",
        )
        result["tuning_elapsed_seconds"] = float(time.time() - tuning_start_time)
        register_tuning_result(results, result)

    if len(results) == 0:
        raise RuntimeError("No stage-1 KNN candidates completed before the tuning deadline.")

    stage_one_results = sorted(results, key=lambda row: row["selection_metric_value"])
    promotion_candidates = stage_one_results[:min(PROMOTION_TOP_N, len(stage_one_results))]

    if len(promotion_candidates) > 0 and not should_stop_tuning(tuning_start_time):
        X_promotion_train, y_promotion_train = sample_aligned_xy(
            X_train,
            y_train,
            PROMOTION_TRAIN_SAMPLE_SIZE,
            "promotion train sample",
        )
        X_promotion_val, y_promotion_val = sample_aligned_xy(
            X_val,
            y_val,
            PROMOTION_VAL_SAMPLE_SIZE,
            "promotion validation sample",
        )

        print(f"Promoting top {len(promotion_candidates)} sampled candidate(s) for larger-sample scoring.")
        for promotion_rank, source_result in enumerate(promotion_candidates, start=1):
            if should_stop_tuning(tuning_start_time):
                break

            trial_index = int(source_result["trial_index"])
            trial_params = source_result["params"]
            print(f"Promotion {promotion_rank}/{len(promotion_candidates)} from trial {trial_index}: {trial_params}")
            result = evaluate_tuning_trial(
                trial_params=trial_params,
                trial_index=trial_index,
                tuner_type="finite_search_promotion",
                X_fit=X_promotion_train,
                y_fit=y_promotion_train,
                X_eval=X_promotion_val,
                y_eval=y_promotion_val,
                eval_prefix="promotion/val_sample",
                tuning_stage="promotion",
            )
            result["promotion_rank"] = promotion_rank
            result["source_trial_index"] = trial_index
            result["tuning_elapsed_seconds"] = float(time.time() - tuning_start_time)
            register_tuning_result(results, result)

    TUNING_RUNTIME_STATE["tuning_elapsed_seconds"] = float(time.time() - tuning_start_time)
    return finalize_tuning_results(results)


### Run Hyperparameter Tuning

Optionally run the staged finite search and activate its best promoted configuration.


In [ ]:
def run_hyperparameter_tuning():
    """Run staged finite tuning or keep the reviewed saved parameters."""
    global MODEL_PARAMS

    TUNING_RUNTIME_STATE.update({
        "unique_candidate_count": None,
        "tuning_elapsed_seconds": None,
        "selected_metric_source": None,
    })

    if not RUN_HYPERPARAMETER_TUNING:
        MODEL_PARAMS = BEST_MODEL_PARAMS.copy()
        print("Hyperparameter tuning disabled. Using BEST_MODEL_PARAMS:")
        print(MODEL_PARAMS)
        return pd.DataFrame()

    return run_finite_search_tuning()

tuning_results = run_hyperparameter_tuning()
print("Active MODEL_PARAMS:", MODEL_PARAMS)


## Fit And Validate

Fit the active model on a bounded training sample and report held-out validation Haversine metrics.


In [ ]:
X_validation_fit, y_validation_fit = sample_aligned_xy(
    X_train,
    y_train,
    PROMOTION_TRAIN_SAMPLE_SIZE,
    "validation fit sample",
)
X_validation_eval, y_validation_eval = sample_aligned_xy(
    X_val,
    y_val,
    PROMOTION_VAL_SAMPLE_SIZE,
    "validation evaluation sample",
)

model = build_model(n_training_samples=len(X_validation_fit))
print(model)

start_time = time.time()
model.fit(X_validation_fit, y_validation_fit)
fit_seconds = time.time() - start_time

val_predictions = predict_in_batches(model, X_validation_eval, PREDICT_BATCH_SIZE, "validation sample")

val_metric_prefix = "validation_sample"
val_metrics, val_distances = evaluate_long_lat_predictions(
    y_validation_eval.values,
    val_predictions,
    prefix=val_metric_prefix,
)

validation_metrics = {
    **val_metrics,
    "fit_seconds": float(fit_seconds),
    "fit_rows": int(len(X_validation_fit)),
    "validation_eval_rows": int(len(X_validation_eval)),
    "metrics_are_sampled": True,
}

print(pd.Series(validation_metrics))

save_best_params_artifact(tuning_results, validation_metrics)
print("Best params JSON written to:", BEST_PARAMS_PATH.resolve())


## Validation Plot

Show the sampled validation mean and Haversine error distribution.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

validation_mean = val_metrics[f"{val_metric_prefix}/haversine_mean_km"]
bars = axes[0].bar(["Validation sample"], [validation_mean], color="#F58518")
axes[0].set_ylabel("Mean Haversine distance (km)")
axes[0].set_title(f"{MODEL_NAME} sampled validation mean error")
axes[0].bar_label(bars, fmt="%.3f km")

axes[1].hist(val_distances, bins=40, color="#F58518")
axes[1].axvline(val_distances.mean(), color="red", linestyle="--", label=f"Mean: {val_distances.mean():.3f} km")
axes[1].set_xlabel("Haversine distance (km)")
axes[1].set_ylabel("Sampled validation trips")
axes[1].set_title(f"{MODEL_NAME} sampled validation error distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

for name in [
    "model",
    "val_predictions",
    "val_distances",
    "X_validation_fit",
    "y_validation_fit",
    "X_validation_eval",
    "y_validation_eval",
    "X_train",
    "y_train",
    "X_val",
    "y_val",
]:
    globals().pop(name, None)
gc.collect()


## Final Fit And Kaggle Submission

Load cached full-train and test matrices, fit the final model, and write the submission file.


In [ ]:
X_final_train, y_final_train, X_final_test, final_gps_scaler, final_categorical_encoder = load_cached_final_data()
X_final_fit, y_final_fit = sample_aligned_xy(
    X_final_train,
    y_final_train,
    FINAL_TRAIN_SAMPLE_SIZE,
    "final train fit sample",
)

final_model = build_model(n_training_samples=len(X_final_fit))
final_model.fit(X_final_fit, y_final_fit)

selected_best_run_summary = BEST_MODEL_RUN_SUMMARY.copy()
selected_best_trial_index = int(selected_best_run_summary["trial_index"])
equivalent_best_trial_indices = selected_best_run_summary.get(
    "equivalent_best_trial_indices",
    [selected_best_trial_index],
).copy()
if "tuning_results" in globals() and not tuning_results.empty:
    if "tuning_stage" in tuning_results.columns and (tuning_results["tuning_stage"] == "promotion").any():
        selected_run_pool = tuning_results[tuning_results["tuning_stage"] == "promotion"]
    else:
        selected_run_pool = tuning_results
    selected_best_run_summary = make_json_safe(
        selected_run_pool.sort_values("selection_metric_value", ascending=True).iloc[0].to_dict()
    )
    selected_best_trial_index = int(selected_best_run_summary["trial_index"])
    equivalent_best_trial_indices = [selected_best_trial_index]

model_serialized = bool(SERIALIZE_KNN_MODEL)
model_bundle = {
    "model": final_model if model_serialized else None,
    "model_serialized": model_serialized,
    "model_omitted_reason": None if model_serialized else "KNN stores the full fit matrix; omitting it keeps the artifact lightweight.",
    "gps_scaler": final_gps_scaler,
    "categorical_encoder": final_categorical_encoder,
    "feature_columns": list(X_final_train.columns),
    "model_params": MODEL_PARAMS.copy(),
    "validation_metrics": validation_metrics.copy(),
    "selected_best_run_summary": selected_best_run_summary,
    "selected_best_trial_index": selected_best_trial_index,
    "equivalent_best_trial_indices": equivalent_best_trial_indices,
    "final_refit": True,
    "final_refit_rows": int(len(X_final_fit)),
    "final_source_train_rows": int(len(X_final_train)),
    "final_train_sample_size": FINAL_TRAIN_SAMPLE_SIZE,
    "predict_batch_size": PREDICT_BATCH_SIZE,
    "runtime_profile": get_runtime_profile(),
    "final_refit_note": "Model is refit on final training data with the selected best parameters; the fitted KNN estimator is serialized only when SERIALIZE_KNN_MODEL is True.",
    "prediction_format": "[LONG, LAT]",
    "preprocessed_data_dir": str(PREPROCESSED_DATA_DIR),
    "preprocessed_schema_version": PREPROCESSED_METADATA.get("schema_version"),
    "preprocessed_metadata": PREPROCESSED_METADATA,
}
joblib.dump(model_bundle, MODEL_BUNDLE_PATH)
print("Model bundle written to:", MODEL_BUNDLE_PATH.resolve())

test_predictions = predict_in_batches(final_model, X_final_test, PREDICT_BATCH_SIZE, "competition test")

submission_path = SUBMISSION_DIR / SUBMISSION_FILE
test_destinations = long_lat_to_lat_long(test_predictions)
write_submission(X_final_test.index, test_destinations, submission_path)
submission = pd.read_csv(submission_path)

print("Submission written to:", submission_path.resolve())
display(submission.head())